In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()

# Test A2-2
kt : VDD3V3

ks : ---> dm.ch1 (I) ---> relay.ch1 (VIN)

ps.ch1 : ---> relay power

ps.ch2 : ---> relay.ch3 (EN)

dm.ch1 : ---> relay.ch1

relay.ch1 : VIN

relay.ch3 : EN

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_2-2] VIN_UVLO, HYS")
log.output_csv(["VIN (V)", "VDD3V3 (V)", "EN (V)", "I_VIN (uA)", "I_VDD3V3 (uA)", "VIN_POK_STS", "SW_STS"])

In [ ]:
disable_all_ps()
relay.init_channels

v_start  = 3.0
v_finish = 4.01

v_vdd = 3.3
v_en  = 3.3

list_temp = list(np.arange(v_start, v_finish, 0.01))
list_vin  = [round(num, 2) for num in list_temp]

print(list_vin)

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

ps.ch2.cfg_all = 3.3, 0.1 # en
ps.ch2.enable
relay.ch3.enable
delay(0.5)

ks.cfg_all = list_vin[0], 0.5 # vin
ks.enable
delay(0.5)

relay.ch1.enable # vin
delay(0.5)

In [ ]:
# ----------------------------------------------------------------------------

count = 0

for vin in list_vin:
    
    ks.vset = vin
    
    i_vin = round(dm.ch1.current_2mA * 1e+6, 6) # uA unit
    i_vdd = round(kt.current * 1e+6, 6) # uA unit
    
    try:
        vin_pok_sts = ic.VIN_POK_STS
        sw_sts = ic.SW_STS
    except:
        vin_pok_sts = "NACK"
        sw_sts = "NACK"
    
    if sw_sts == 2:
        count += 1
    
    ret = [vin, v_vdd, v_en, i_vin, i_vdd, vin_pok_sts, sw_sts]
    log.output_csv(ret)
    
    print(ret)
    
    if count == 10:
        break

# ----------------------------------------------------------------------------

count = 0

for vin in reversed(list_vin):
    
    ks.vset = vin
    
    i_vin = round(dm.ch1.current_2mA * 1e+6, 6) # uA unit
    i_vdd = round(kt.current * 1e+6, 6) # uA unit
    
    try:
        vin_pok_sts = ic.VIN_POK_STS
        sw_sts = ic.SW_STS
    except:
        vin_pok_sts = "NACK"
        sw_sts = "NACK"
    
    if sw_sts == 0:
        count += 1
    
    ret = [vin, v_vdd, v_en, i_vin, i_vdd, vin_pok_sts, sw_sts]
    log.output_csv(ret)
    
    print(ret)
    
    if count == 10:
        break
    
# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()